# 🧪 KIỂM THỬ MÔ HÌNH HỌC MÁY PHÂN TÁN SPARK MLLIB ALS
**Học phần:** Nhập môn Big Data (HUIT) - Mã HP: 01200057  
**Mục tiêu:** Nạp mô hình `als_model_final` đã huấn luyện, thực hiện suy diễn gợi ý sản phẩm cá nhân hóa cho khách hàng cũ và kiểm thử chiến lược Hybrid Fallback (Cold-Start) cho khách hàng mới.

### Nhóm thực hiện:
1. Tăng Gia Huy (2001230313) - Hạ tầng HDFS & Ingestion Streaming
2. Trần Thị Minh Phương (2001230704) - Xử lý Phân tán & Spark ETL
3. Trần Bảo Nguyên (2001230587) - Spark MLlib ALS & Benchmark
4. Lê Ngọc Anh (2001230017) - Airflow DAG & Serving UI

In [ ]:
import os
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALSModel

# Khởi tạo Spark Session cục bộ
spark = SparkSession.builder \
    .appName("Notebook_Test_ALS") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("✅ Spark Session đã sẵn sàng!")

## 1. Nạp mô hình đã huấn luyện từ thư mục models/

In [ ]:
model_path = "../models/als_model_final"
if not os.path.exists(model_path):
    model_path = "../models/als_model"

model = ALSModel.load(model_path)
print(f"✅ Đã nạp thành công ALS Model từ đường dẫn: {model_path}")
print(f"Số chiều ẩn (Latent Factors / Rank): {model.rank}")

## 2. Thử nghiệm gợi ý cá nhân hóa cho Khách hàng cũ (VD: User 147545)

In [ ]:
test_user_id = 147545
user_df = spark.createDataFrame([(test_user_id,)], ["visitorid"])

num_recommendations = 5
user_recs = model.recommendForUserSubset(user_df, num_recommendations)

print(f"Kết quả dự đoán Top {num_recommendations} sản phẩm cho Khách hàng #{test_user_id}:")
user_recs.show(truncate=False)

## 3. Thử nghiệm xử lý bài toán Cold-Start cho Khách hàng mới (Hybrid Fallback)

In [ ]:
new_user_id = 999999
new_user_df = spark.createDataFrame([(new_user_id,)], ["visitorid"])
new_user_recs = model.recommendForUserSubset(new_user_df, 5)

if new_user_recs.count() == 0:
    print(f"⚠️ Khách hàng {new_user_id} chưa có trong ma trận (Cold Start)!")
    print("🌟 Tự động kích hoạt Hybrid Fallback: Trích xuất Top sản phẩm thịnh hành từ Spark SQL:")
    
    top_csv_dir = "../models/top_products_dir"
    top_df = None
    if os.path.exists(top_csv_dir):
        for f in os.listdir(top_csv_dir):
            if f.endswith('.csv'):
                top_df = pd.read_csv(os.path.join(top_csv_dir, f))
                break
    
    if top_df is not None:
        display(top_df.head(5))
    else:
        print("Dùng danh sách Top 5 mặc định: [187946, 461686, 5411, 370653, 219512]")